In [ ]:
import jax
import jax.numpy as jnp

def sample_actions(act_logits, random_key, task, batch_size=1):
    """
    act_logits:
       For "sampling": shape (parallel_env, 1, 2*action_dim)
       For "batch": same input shape; output will have an extra sample axis.
    random_key: jax PRNG key
    task: "sampling" or "batch"
    batch_size: number of samples per env (only used in "batch" branch)
    """
    if task == "batch":
        batch_size = 1  # number of samples per env
        action_dim = act_logits.shape[-1] // 2

        act_logits  = act_logits = jnp.repeat(act_logits, batch_size, axis=2)
        means = act_logits[..., :action_dim]
        log_stds = act_logits[..., action_dim:]
        log_stds = jnp.clip(log_stds, -20.0, 2.0)
        stds = jnp.exp(log_stds)
        # print("my ass", log_stds, means)
        # 4. Sample noise and compute actions:
        noise = jax.random.normal(random_key, shape=means.shape)  # shape: (parallel_env, batch_size, action_dim)
        # print("n", noise.shape, noise)
        acts_tick = means + noise * stds  # shape: (parallel_env, batch_size, action_dim)

    elif task == "sampling":
        action_dim = act_logits.shape[-1] // 2
        means = act_logits[..., :action_dim].squeeze(-1)
        log_stds = act_logits[..., action_dim:].squeeze(-1)
        # print("policy_out", act_logits.shape, "mean", means.shape, "std", log_stds.shape)
        
        # Clip log_stds for numerical stability
        log_stds = jnp.clip(log_stds, -20.0, 2.0)
        stds = jnp.exp(log_stds)
        # print("my pus", log_stds, means)
        # Sample from standard normal and scale
        noise = jax.random.normal(random_key, means.shape)
        # print("f", noise.shape, noise)
        acts_tick = means + noise * stds
        # print(acts_tick.shape)
        # jax.debug.print("dit kan echt niet meer {} {} {} ", means.shape, log_stds.shape, acts_tick.shape)
        acts_tick = jnp.squeeze(acts_tick)
        

    else:
        raise ValueError("Invalid task. Choose either 'batch' or 'sampling'.")
    return acts_tick

def gaussian_log_prob(actions, act_logits, task):
    print(act_logits.shape, actions.shape)
    """
    actions:
       For "sampling": shape (time, env_steps, action_dim)  (e.g., (1, 256, 1))
       For "batch": if actions lack a sample axis, one will be added.
    act_logits:
       For "sampling": shape (time, env_steps, 1, 2*action_dim)
       For "batch": same expected shape.
    Returns:
       Gaussian log-probabilities. For "batch" mode, the log-probabilities are computed per sample
       (optionally summed over the action dimension).
    """
    if task == "sampling":
        action_dim = act_logits.shape[-1] // 2
        means = act_logits[..., :action_dim].squeeze()
        log_stds = act_logits[..., action_dim:].squeeze()
        
        
        # Clip log_stds for numerical stability
        log_stds = jnp.clip(log_stds, -20.0, 2.0)
        
        
        variance = jnp.exp(2 * log_stds)
        # print("variance", variance, "means", means, "log_stds", log_stds)
        
        log_prob = -0.5 * (
            jnp.log(2 * jnp.pi)
            + 2 * log_stds
            + (actions - means) ** 2 / variance
                    )

    elif task == "batch":
        action_dim = act_logits.shape[-1] // 2
        batch_size = 1
        act_logits = jnp.broadcast_to(act_logits, (act_logits.shape[0], act_logits.shape[1], batch_size, act_logits.shape[2]))
        act_logits = jnp.repeat(act_logits, batch_size, axis=2)
        means = act_logits[..., :action_dim]
        log_stds = act_logits[..., action_dim:]
        
        
    
        log_stds = jnp.clip(log_stds, -20.0, 2.0)
        variance = jnp.exp(2 * log_stds)
        # print("variance", variance, "means", means, "log_stds", log_stds)
        
        log_prob = -0.5 * (jnp.log(2 * jnp.pi) + 2 * log_stds + ((actions - means) ** 2) / variance)
        log_prob = jnp.squeeze(log_prob.sum(axis=2), axis=-1)

    else:
        raise ValueError("Invalid task. Choose either 'batch' or 'sampling'.")
    return log_prob




In [86]:
# -------------------------------
# Testing the functionality

key = jax.random.PRNGKey(0)

# print("=== Testing sample_actions ===")
# # For "sampling" branch:
# # Input: (parallel_env, 1, 2*action_dim) e.g. (8, 1, 2) with action_dim=1.
# # act_logits_sampling = jax.random.normal(key, shape=(8, 1, 2))
# act_logits_sampling = jax.numpy.ones((8, 1, 2))
# print("act_logits_sampling", act_logits_sampling.shape)
# # key, subkey = jax.random.split(key)
# sampled_sampling = sample_actions(act_logits_sampling, key, task="sampling")
# # print("Sampling branch output (should be shape (8,) or (8,1) after squeezing):", sampled_sampling.shape)
# # print("Sampling branch output:", sampled_sampling)

# # For "batch" branch with batch_size = 1:
# # key, subkey = jax.random.split(key)
# sampled_batch = sample_actions(act_logits_sampling, key, task="batch", batch_size=1)
# # Remove singleton sample dimension for comparison:
# # sampled_batch_squeezed = jnp.squeeze(sampled_batch)
# # print("Batch branch (batch_size=1) output (after squeezing, should be same as sampling):", sampled_batch_squeezed.shape)
# # print("Batch branch output:", sampled_batch_squeezed)
# print(sampled_sampling, "\n", sampled_batch)

print("\n=== Testing gaussian_log_prob ===")
# For "sampling" branch log_prob:
# Let act_logits have shape (time, env_steps, 1, 2*action_dim), e.g. (1, 256, 1, 2)
# And actions shape (1, env_steps, 1), e.g. (1, 256, 1)
env_steps = 8
# act_logits_log_prob = jax.random.normal(key, shape=(1, env_steps, 1, 2))
# actions_log_prob = jax.random.normal(key, shape=(1, env_steps))
act_logits_log_prob = jax.numpy.ones((1, env_steps, 2))
actions_log_prob = jax.numpy.ones((1, env_steps))
log_prob_sampling = gaussian_log_prob(actions_log_prob, act_logits_log_prob, task="sampling")
print("Gaussian log-prob (sampling) output shape:", log_prob_sampling)

# For "batch" branch log_prob:
act_logits_log_prob = jax.numpy.ones((1, env_steps, 1, 2))
actions_log_prob = jax.numpy.ones((1, env_steps, 1, 1))
act_logits_log_prob = jax.random.normal(key, shape=(1, env_steps, 1, 2))
actions_log_prob = jax.random.normal(key, shape=(1, env_steps, 1, 1))
log_prob_batch = gaussian_log_prob(actions_log_prob, act_logits_log_prob, task="batch")
print("Gaussian log-prob (batch) output shape:", log_prob_batch)

# Optionally, if you want to aggregate the samples (e.g., by summing over the sample axis),
# you could do:
# if log_prob_batch.ndim > 3:  # if sample axis exists
#     log_prob_batch_aggregated = log_prob_batch.sum(axis=2)
#     print("Gaussian log-prob (batch) aggregated over samples shape:", log_prob_batch_aggregated.shape)


=== Testing gaussian_log_prob ===
(1, 8, 2) (1, 8)
Gaussian log-prob (sampling) output shape: [[-1.9189385 -1.9189385 -1.9189385 -1.9189385 -1.9189385 -1.9189385
  -1.9189385 -1.9189385]]
(1, 8, 1, 2) (1, 8, 1, 1)
Gaussian log-prob (batch) output shape: [[  -2.9189386    -4.3780413    -1.2456048    -1.4456145    -0.76598185
  -246.85428      -1.3426186    -2.4443378 ]]


In [56]:
import jax.numpy as jnp
import numpy as np

# Example array of shape (1, 256, 1, 2)
array = np.random.rand(1, 2, 1, 2)  # Replace with your actual array
print(array)
# Desired expansion factor for the 3rd dimension (e.g., x = 4)
x = 4  

# Expand along the third axis
expanded_array = np.tile(array, (1, 1, x, 1))
a = np.repeat(array, x, axis=2)
print(a)
print(expanded_array.shape)  # Should output (1, 256, x, 2)

print(array,"\n", expanded_array)


[[[[0.7275847  0.898863  ]]

  [[0.26442263 0.96225166]]]]
[[[[0.7275847  0.898863  ]
   [0.7275847  0.898863  ]
   [0.7275847  0.898863  ]
   [0.7275847  0.898863  ]]

  [[0.26442263 0.96225166]
   [0.26442263 0.96225166]
   [0.26442263 0.96225166]
   [0.26442263 0.96225166]]]]
(1, 2, 4, 2)
[[[[0.7275847  0.898863  ]]

  [[0.26442263 0.96225166]]]] 
 [[[[0.7275847  0.898863  ]
   [0.7275847  0.898863  ]
   [0.7275847  0.898863  ]
   [0.7275847  0.898863  ]]

  [[0.26442263 0.96225166]
   [0.26442263 0.96225166]
   [0.26442263 0.96225166]
   [0.26442263 0.96225166]]]]
